<a id="top"></a>

# Lab L1005: tool failures are messages, not exceptions

```yaml
title: "Lab L1005: tool failures are messages, not exceptions"
keywords: tool failure, ToolNode, handle_tool_errors, ToolMessage, status error, error as data, back-edge, recovery, GraphRecursionError, FakeModel, offline, lab
estimated duration: 35
```

> **Lesson:** L10. See [objectives.md](../../../../docs/origin/lesson_roadmaps/L10/objectives.md).
> Solutions copy — every `# TODO` filled in. Your copy: `L1005_lab_empty.ipynb`. **No API key: pure
> Python.** You drive a tool that *raises* through the graph and watch `ToolNode` turn the crash into
> a message.
>
> **Builds on [L08](../L08/objectives.md):** L08 taught the *tool author* what to **return** on
> failure (errors as data). This lab is the *graph* layer — what happens when the tool **can't even
> return** (it raises), and how the `ToolNode(handle_tool_errors=...)` knob decides crash vs. recover.
>
> **You'll walk out able to** toggle `handle_tool_errors` and explain the difference, watch a raised
> exception become a `ToolMessage(status="error")` that flows back through the back-edge, and
> distinguish a *raise* from an *error-as-data* return.

## Contents

- [1. Setup](#1-setup)
- [2. Problem 1 — handle_tool_errors=False: one bad tool crashes the agent](#2-problem-1--handle_tool_errorsfalse-one-bad-tool-crashes-the-agent)
- [3. Problem 2 — handle_tool_errors=True: the crash becomes a message](#3-problem-2--handle_tool_errorstrue-the-crash-becomes-a-message)
- [4. Problem 3 — error-as-data needs no graph change](#4-problem-3--error-as-data-needs-no-graph-change)
- [5. Problem 4 — Why not dump the traceback? (written)](#5-problem-4--why-not-dump-the-traceback-written)
- [6. Problem 5 — Should the graph auto-retry? (written)](#6-problem-5--should-the-graph-auto-retry-written)
- [7. Self-check](#7-self-check)

## 1. Setup

All handed to you: `lookup` and `flaky_fetch` (its URL decides its behavior — `https://ok` works,
`https://error` returns an error *as data*, `https://crash` *raises*), the scripted `FakeModel`, the
`AgentState` + `route` from L1004, and a `build_agent` that now exposes `ToolNode`'s
**`handle_tool_errors`** knob, plus a `describe` helper to print messages. **Your job: script and run
the three failure scenarios.**

In [1]:
from typing import Annotated, Any, TypedDict

from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, ToolMessage
from langgraph.graph import END, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

from fluffy_potato_curriculum.common.fake_model import (
    FakeModel,
    text_reply,
    tool_call,
    tool_reply,
)


def lookup(city: str) -> str:
    """Return a city's population, or raise KeyError if it isn't in the table."""
    populations = {"tokyo": "37000000", "lagos": "15000000", "paris": "11000000"}
    key = city.strip().lower()
    if key not in populations:
        raise KeyError(f"no population on file for {city!r}")
    return populations[key]


def flaky_fetch(url: str) -> str:
    """A tiny fake network keyed by URL, so each failure mode is reproducible:

    - https://ok      -> a usable value (success)
    - https://error   -> a STRUCTURED error returned as data (the L08 pattern; no exception)
    - https://crash   -> RAISES RuntimeError (an uncaught exception the graph must handle)
    """
    if url == "https://ok":
        return "the answer is 42"
    if url == "https://error":
        return '{"error": "503 service unavailable"}'
    if url == "https://crash":
        raise RuntimeError("connection reset by peer")
    raise ValueError(f"unknown url: {url!r}")


TOOLS = [lookup, flaky_fetch]


class AgentState(TypedDict):
    """add_messages APPENDS each node's messages to the running conversation."""

    messages: Annotated[list[BaseMessage], add_messages]


def route(state: AgentState) -> str:
    """Conditional edge from L1004: tool calls -> 'tools'; plain text -> END."""
    last = state["messages"][-1]
    if isinstance(last, AIMessage) and last.tool_calls:
        return "tools"
    return END


def build_agent(model: Any, *, handle_tool_errors: bool = True) -> Any:
    """The L1004 graph, now exposing ToolNode's `handle_tool_errors` knob (the point of this lab)."""

    async def agent_node(state: AgentState) -> dict[str, list[BaseMessage]]:
        return {"messages": [await model.bind_tools(TOOLS).ainvoke(state["messages"])]}

    builder = StateGraph(AgentState)
    builder.add_node("agent", agent_node)
    builder.add_node("tools", ToolNode(TOOLS, handle_tool_errors=handle_tool_errors))
    builder.set_entry_point("agent")
    builder.add_conditional_edges("agent", route, {"tools": "tools", END: END})
    builder.add_edge("tools", "agent")
    return builder.compile()


def describe(msg: BaseMessage) -> str:
    """One readable line per message: type, tool calls, tool results (with status), or text."""
    kind = type(msg).__name__
    if isinstance(msg, AIMessage) and msg.tool_calls:
        calls = ", ".join(f"{c['name']}({c['args']})" for c in msg.tool_calls)
        return f"{kind:12} -> tool call: {calls}"
    if isinstance(msg, ToolMessage):
        return f"{kind:12} <- result [{msg.status}]: {msg.content!r}"
    return f"{kind:12}    {str(msg.content)!r}"

[↑ Back to top](#top)

## 2. Problem 1 — `handle_tool_errors=False`: one bad tool crashes the agent

Build the graph with `handle_tool_errors=False`. Script the model to call
`flaky_fetch("https://crash")`, which **raises** `RuntimeError`. `await` the `ainvoke` inside a
`try/except RuntimeError` and prove the exception **escaped the graph** — one buggy tool, one dead
run.

In [2]:
# flaky_fetch("https://crash") RAISES. With handle_tool_errors=False the exception escapes the
# tools node and kills the whole ainvoke -- one buggy tool crashes the agent.
crash_script = [
    tool_reply(tool_call("x1", "flaky_fetch", {"url": "https://crash"})),
    text_reply("unreachable -- the graph already crashed"),
]
strict_graph = build_agent(FakeModel(crash_script), handle_tool_errors=False)

escaped = False
try:
    await strict_graph.ainvoke({"messages": [HumanMessage(content="Fetch https://crash")]})
except RuntimeError as exc:
    escaped = True
    print("the exception escaped the graph:", repr(exc))
assert escaped

the exception escaped the graph: RuntimeError('connection reset by peer')


[↑ Back to top](#top)

## 3. Problem 2 — `handle_tool_errors=True`: the crash becomes a message

Same crashing call — now build with `handle_tool_errors=True`. `ToolNode` catches the
`RuntimeError` and appends a `ToolMessage(status="error")`; the **back-edge** hands it to the model,
which retries `https://ok` and finishes. Confirm the error `ToolMessage` shows up and the run reaches
natural termination instead of crashing.

In [3]:
# Same crashing call, but handle_tool_errors=True: ToolNode catches the RuntimeError and appends a
# ToolMessage(status="error"). The back-edge hands that message to the model, which retries a good
# url and finishes -- natural termination, no crash.
recover_script = [
    tool_reply(tool_call("x1", "flaky_fetch", {"url": "https://crash"})),  # raises -> error message
    tool_reply(tool_call("x2", "flaky_fetch", {"url": "https://ok"})),  # recover
    text_reply("https://crash failed, so I fetched https://ok: the answer is 42."),
]
safe_graph = build_agent(FakeModel(recover_script), handle_tool_errors=True)
result = await safe_graph.ainvoke(
    {"messages": [HumanMessage(content="Fetch https://crash, else https://ok")]}
)
for msg in result["messages"]:
    print(describe(msg))

error_msgs = [m for m in result["messages"] if isinstance(m, ToolMessage) and m.status == "error"]
last = result["messages"][-1]
assert error_msgs  # the crash became an error ToolMessage, not a crash
assert isinstance(last, AIMessage) and not last.tool_calls  # natural termination

HumanMessage    'Fetch https://crash, else https://ok'
AIMessage    -> tool call: flaky_fetch({'url': 'https://crash'})
ToolMessage  <- result [error]: "Error: RuntimeError('connection reset by peer')\n Please fix your mistakes."
AIMessage    -> tool call: flaky_fetch({'url': 'https://ok'})
ToolMessage  <- result [success]: 'the answer is 42'
AIMessage       'https://crash failed, so I fetched https://ok: the answer is 42.'


[↑ Back to top](#top)

## 4. Problem 3 — error-as-data needs no graph change

`flaky_fetch("https://error")` **returns** a structured error string — it never raises. The tool
already followed L08's "errors are values" pattern, so the graph needs **zero** changes: the content
flows back as an ordinary `ToolMessage` (status `"success"` — the *call* succeeded; the *content*
says it failed), and the model reads it and recovers. Run it and compare the status against
Problem 2's raising case.

In [4]:
# error-as-data: flaky_fetch("https://error") RETURNS a structured error string (it does not raise).
# The tool followed L08's "errors are values" pattern, so the graph needs NO change -- the content
# just flows back as an ordinary (status="success") ToolMessage, and the model reads it and recovers.
data_script = [
    tool_reply(
        tool_call("e1", "flaky_fetch", {"url": "https://error"})
    ),  # returns '{"error": ...}'
    tool_reply(tool_call("e2", "flaky_fetch", {"url": "https://ok"})),  # recover
    text_reply("https://error returned a 503, so I used https://ok: the answer is 42."),
]
data_graph = build_agent(FakeModel(data_script), handle_tool_errors=True)
data_result = await data_graph.ainvoke(
    {"messages": [HumanMessage(content="Fetch https://error, else https://ok")]}
)
for msg in data_result["messages"]:
    print(describe(msg))

# The error-as-data ToolMessage came back with status "success" -- the CALL succeeded; the CONTENT
# says it failed. That is the distinction from the raising case in Problem 2.
first_tool_msg = next(m for m in data_result["messages"] if isinstance(m, ToolMessage))
print("\nerror-as-data status:", first_tool_msg.status)  # "success" -- no exception was raised
assert first_tool_msg.status == "success"

HumanMessage    'Fetch https://error, else https://ok'
AIMessage    -> tool call: flaky_fetch({'url': 'https://error'})
ToolMessage  <- result [success]: '{"error": "503 service unavailable"}'
AIMessage    -> tool call: flaky_fetch({'url': 'https://ok'})
ToolMessage  <- result [success]: 'the answer is 42'
AIMessage       'https://error returned a 503, so I used https://ok: the answer is 42.'

error-as-data status: success


[↑ Back to top](#top)

## 5. Problem 4 — Why not dump the traceback? (written)

`ToolNode`'s error message is short — a class name plus one line, not the full Python traceback.
Give **two** reasons that's the right call. (Hint: think tokens, what the model can act on, and what
a stack trace might leak.)

_Write your answer by editing this cell (double-click)._

1. **Tokens / noise.** A 40-line traceback is expensive and mostly irrelevant to the model — it
can't act on file paths and frame internals. A short `RuntimeError('connection reset by peer')` is
the actionable signal.
2. **Leakage.** A stack trace exposes internal paths, module layout, and implementation details you'd
rather not surface to the model — or to a user who might see the output.

[↑ Back to top](#top)

## 6. Problem 5 — Should the graph auto-retry? (written)

`ToolNode` does **not** auto-retry — it surfaces the error and lets the model decide. In a sentence
or two: why is blanket auto-retry a poor default? (Hint: a `404 not found` isn't a `503 service
unavailable`; picture a tool that charges a card.)

_Write your answer by editing this cell (double-click)._

**Answer:** Failures aren't alike — a `404 row not found` will never succeed on retry, while a `503`
might; blind retries waste tokens and can mask real bugs. Worse, a non-idempotent tool (one that
charges a card or sends an email) makes auto-retry actively dangerous. Retry is a deliberate
choice — with its own budget and idempotency guarantees — never a reflex; the safe default is
surface-and-let-the-model-decide.

[↑ Back to top](#top)

## 7. Self-check

Compare against `L1005_lab_empty.ipynb`. Done when: with `handle_tool_errors=False` the
`RuntimeError` escapes `ainvoke`; with `handle_tool_errors=True` the same crash becomes a
`ToolMessage(status="error")` and the run reaches natural termination; the error-as-data call comes
back with status `"success"` and needs no graph change; and you can say why a short error beats a
traceback and why auto-retry is not a safe default.

[↑ Back to top](#top)